# 3 · Comparing Across Categories
*Data Visualization for Scientists & Public Health Professionals*

A large share of public-health questions are comparisons across groups: how many encounters in each specialty? does length of stay differ by age? how does readmission split by treatment? The **bar chart family** answers these. This notebook covers counting categories, ordering bars so they can be read, showing an aggregate per category, splitting into subgroups, going horizontal when there are many categories, and the one rule that keeps a bar chart honest — a rule the capstone at the end puts straight to work on the rural-closures data.

### Learning objectives
- Count occurrences per category with a count plot
- Order bars for legibility — and know when *not* to
- Show a mean per category with a bar plot, and read its error bars
- Split a comparison into subgroups with grouped (hue) bars
- Use horizontal bars for many categories or long labels
- Keep a bar chart honest with a zero baseline

### Agenda
1. Counting categories
2. Ordering matters (and when it doesn't)
3. Aggregate bars
4. Grouped bars
5. Horizontal bars for many categories
6. Honest bars
7. **Capstone Part 1 — count and compare the closures**

### How the exercises work
Each exercise has a prompt, an empty cell to try it yourself, and a collapsed **Solution** you can expand to check your work.

## Setup

Same `diabetes_viz` data for the teaching sections. We fix the natural order of the age bands once, since one of the lessons below is about *respecting* an order that already exists.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

BASE_URL = "https://raw.githubusercontent.com/jimcody2014/2026-python-data/refs/heads/main"
df = pd.read_csv(f"{BASE_URL}/diabetes_viz.csv", low_memory=False)

age_order = ["[0-10)", "[10-20)", "[20-30)", "[30-40)", "[40-50)",
             "[50-60)", "[60-70)", "[70-80)", "[80-90)", "[90-100)"]
df.shape

## 1. Counting categories

The simplest categorical chart counts how many rows fall in each category. `sns.countplot` does this directly — think of it as a histogram for a categorical variable. Here: how many encounters for each `race`?

In [ ]:
fig, ax = plt.subplots(figsize=(8, 4))
sns.countplot(data=df, x="race", color="steelblue", ax=ax)
ax.set_title("Encounters by race")
ax.set_xlabel("Race")
ax.set_ylabel("Encounters")
ax.tick_params(axis="x", rotation=30)
fig.tight_layout()
plt.show()

> **Tip:** `countplot` counts the rows for you. If you already have counts (say from `value_counts`), use `barplot` on those values instead. [seaborn categorical tutorial](https://seaborn.pydata.org/tutorial/categorical.html)

## 2. Ordering matters (and when it doesn't)

By default the bars appear in whatever order the categories happen to occur, which forces the reader to hunt for the biggest and smallest. For an **unordered** category, sort the bars by value — it is one of the cheapest readability wins there is. Pass `order=` a list of categories in the order you want.

In [ ]:
by_freq = df["race"].value_counts().index   # categories, most frequent first

fig, axes = plt.subplots(1, 2, figsize=(12, 4), sharey=True)

sns.countplot(data=df, x="race", color="steelblue", ax=axes[0])
axes[0].set_title("Default order")
axes[0].set_xlabel("Race")
axes[0].tick_params(axis="x", rotation=30)

sns.countplot(data=df, x="race", order=by_freq, color="steelblue", ax=axes[1])
axes[1].set_title("Ordered by frequency")
axes[1].set_xlabel("Race")
axes[1].tick_params(axis="x", rotation=30)

fig.tight_layout()
plt.show()

> **Note:** The right chart answers "which group is largest?" at a glance; the left makes you read every bar. The exception is a category with a **natural** order — age bands, severity levels — where sorting by value would destroy the very pattern you want to see. We meet that case in the next section.

## 3. Aggregate bars

A `countplot` counts rows. A **`barplot`** summarizes a numeric column per category — by default the **mean** — and draws a thin error bar for a 95% confidence interval of that mean. Here: mean length of stay for each age band. Age is naturally ordered, so we pass `age_order` and **keep** that order rather than sorting by height — sorting would scramble the age axis and hide the trend.

In [ ]:
fig, ax = plt.subplots(figsize=(9, 4))
sns.barplot(data=df, x="age", y="time_in_hospital", order=age_order, color="steelblue", ax=ax)
ax.set_title("Mean length of stay by age (natural order kept)")
ax.set_xlabel("Age band")
ax.set_ylabel("Mean days in hospital")
ax.tick_params(axis="x", rotation=45)
fig.tight_layout()
plt.show()

> **Note:** The error bar shows uncertainty in the **mean**, not the spread of the data — a common misread. With tens of thousands of rows the bars are tiny, which itself tells you the means are precisely estimated. Pass `errorbar=None` to drop them, or `estimator="median"` to summarize differently. The rising staircase is the payoff of keeping age in order. [seaborn barplot](https://seaborn.pydata.org/generated/seaborn.barplot.html)

### Exercise 1 — An ordered aggregate bar *(6 min)*

Draw a bar plot of the **mean `num_lab_procedures` for each `race`**, ordered by the mean (smallest to largest), fully labeled. Because `race` is unordered, sorting by value here is the right move — the opposite of what you just did with age.

In [ ]:
# Your work here


<details>
<summary><b>Solution</b></summary>

```python
order = df.groupby("race")["num_lab_procedures"].mean().sort_values().index

fig, ax = plt.subplots(figsize=(8, 4))
sns.barplot(data=df, x="race", y="num_lab_procedures", order=order, color="steelblue", ax=ax)
ax.set_title("Mean lab procedures by race")
ax.set_xlabel("Race")
ax.set_ylabel("Mean lab procedures")
ax.tick_params(axis="x", rotation=30)
fig.tight_layout()
plt.show()
```

**Why this works.** `barplot` computes the per-group mean for you; building `order` from the same grouped means sorts the bars by height. Unlike age, `race` carries no inherent sequence, so ordering by value is exactly what makes the comparison legible. The means turn out close — a reminder to read the y-axis before declaring a difference, which Section 6 makes vivid.

</details>

## 4. Grouped bars

To bring in a *second* categorical dimension, pass `hue=`. Each category on the x-axis splits into one bar per hue level, drawn side by side. Here: within each readmission outcome, how many patients were versus were not prescribed a diabetes medication?

In [ ]:
fig, ax = plt.subplots(figsize=(9, 4))
sns.countplot(data=df, x="readmitted", order=["NO", ">30", "<30"],
              hue="diabetesMed", ax=ax)
ax.set_title("Readmission, split by whether a diabetes med was prescribed")
ax.set_xlabel("Readmitted")
ax.set_ylabel("Encounters")
ax.legend(title="Diabetes med")
fig.tight_layout()
plt.show()

> **Tip:** Keep the number of hue levels small — two to four. Beyond that the groups blur together, and a set of small multiples (the faceting lesson in Session 2) reads far better.

## 5. Horizontal bars for many categories

With many categories or long labels, vertical bars collide and their text rotates into an unreadable tangle. Putting the category on the **y-axis** gives every label its own line. There are 72 specialties here, so we order by frequency and show the ten busiest.

In [ ]:
top_spec = df["medical_specialty"].value_counts().head(10).index

fig, ax = plt.subplots(figsize=(8, 5))
sns.countplot(data=df, y="medical_specialty", order=top_spec, color="steelblue", ax=ax)
ax.set_title("Ten busiest specialties")
ax.set_xlabel("Encounters")
ax.set_ylabel("Specialty")
fig.tight_layout()
plt.show()

> **Note:** Switching from `x=` to `y=` is the whole trick. Ordering then reads naturally from the longest bar at the top to the shortest at the bottom — try to imagine these ten labels crammed onto a vertical axis and the point makes itself.

### Exercise 2 — Sort one, leave the other *(8 min)*

Build a figure with two panels. **Left:** a horizontal count plot of the **top 10 `medical_specialty`** values, ordered most-to-least. **Right:** a bar of **mean `num_medications` by `age`**, left in natural age order. In a comment, explain why you sorted the first but not the second.

In [ ]:
# Your work here


<details>
<summary><b>Solution</b></summary>

```python
fig, axes = plt.subplots(1, 2, figsize=(13, 5))

top10 = df["medical_specialty"].value_counts().head(10).index
sns.countplot(data=df, y="medical_specialty", order=top10, color="steelblue", ax=axes[0])
axes[0].set_title("Ten busiest specialties")
axes[0].set_xlabel("Encounters")
axes[0].set_ylabel("Specialty")

sns.barplot(data=df, x="age", y="num_medications", order=age_order, color="steelblue", ax=axes[1])
axes[1].set_title("Mean medications by age")
axes[1].set_xlabel("Age band")
axes[1].set_ylabel("Mean medications")
axes[1].tick_params(axis="x", rotation=45)

fig.tight_layout()
plt.show()
# Specialty is unordered, so sorting by count makes the ranking readable.
# Age has a natural order, so we keep it -- sorting by value would destroy the age trend.
```

**Why this works.** The two panels are the two halves of the ordering rule in one figure. `medical_specialty` has no inherent sequence, so a value sort turns a jumble into a ranking; `age` is inherently ordered, so we preserve it and let the trend show. Same `order=` argument, opposite decision — driven entirely by whether the category is ordered.

</details>

## 6. Honest bars

A bar encodes its value as a **length**, so the axis must start at **zero** — otherwise the lengths stop matching the numbers and a trivial difference can be made to look enormous. Women and men in this data average almost exactly the same number of lab procedures (43.15 versus 43.03). Watch what a truncated axis does to that non-difference.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(11, 4))

sns.barplot(data=df, x="gender", y="num_lab_procedures", errorbar=None, color="steelblue", ax=axes[0])
axes[0].set_ylim(42.8, 43.3)   # truncated baseline manufactures a gap
axes[0].set_title("Misleading: y-axis starts at 42.8")
axes[0].set_xlabel("Gender")
axes[0].set_ylabel("Mean lab procedures")

sns.barplot(data=df, x="gender", y="num_lab_procedures", errorbar=None, color="steelblue", ax=axes[1])
axes[1].set_title("Honest: y-axis starts at 0")
axes[1].set_xlabel("Gender")
axes[1].set_ylabel("Mean lab procedures")

fig.tight_layout()
plt.show()

> **Warning:** The left chart invents a striking gender gap in testing; the right one tells the truth, that there is essentially none. A fabricated disparity about *people* is exactly the kind of thing a truncated bar axis produces — so whenever you see a bar chart, check the baseline before you believe the story. Two habits keep bars honest: start lengths at zero, and pick the summary that fits the data (a median for a skewed variable, not the mean).

## Wrap-up

You can now count categories with a count plot, order bars for legibility while leaving naturally ordered categories alone, summarize a numeric value per category with a bar plot and read its error bars, split a comparison with hue, go horizontal for many or long-labeled categories, and keep a bar honest with a zero baseline. Time to use all of it on real stakes.

---

## Capstone · Part 1 — Count and compare the closures

Everything above was practice on the diabetes data. Now we turn to the project that runs through both courses: **rural hospital closures**. You cleaned and analyzed this data in the pandas course; here you begin to *show* it — and this section uses only the skills from Notebooks 2 and 3.

**Driving question.** Which states and payment models bore the brunt of rural hospital closures — and were the hospitals that closed big or small?

We start from the cleaned data. The cell below reproduces, in one step, the tidy-up you did in the pandas capstone — dropping the saved index, fixing the mistyped closure year, standardizing `closure_type`, filling the few missing bed counts, removing bed-count outliers with the 1.5 × IQR rule, and dropping rows with no state — so we begin analysis-ready and spend our time on the charts.

In [ ]:
closures = pd.read_csv(f"{BASE_URL}/rural_hospital_closures.csv")

closures = closures.drop(columns="Unnamed: 0")
closures["closure_year"] = closures["closure_year"].replace(1019, 2019)
closures["closure_type"] = (closures["closure_type"].str.strip().str.capitalize()
                            .replace({"Complet": "Complete", "Convertd": "Converted"}))
closures["beds"] = closures["beds"].fillna(closures["beds"].median())
q1, q3 = closures["beds"].quantile([0.25, 0.75])
iqr = q3 - q1
closures = closures[closures["beds"].between(q1 - 1.5 * iqr, q3 + 1.5 * iqr)]
closures = closures.dropna(subset=["state"])
closures.shape

### Task 1 — Which states lost the most?

Make a horizontal count plot of the **top 10 states** by number of closures, ordered most-to-least, fully labeled.

In [ ]:
# Your work here


<details>
<summary><b>Solution</b></summary>

```python
top_states = closures["state"].value_counts().head(10).index

fig, ax = plt.subplots(figsize=(8, 5))
sns.countplot(data=closures, y="state", order=top_states, color="steelblue", ax=ax)
ax.set_title("Rural hospital closures by state (top 10)")
ax.set_xlabel("Closures")
ax.set_ylabel("State")
fig.tight_layout()
plt.show()
# Texas, Oklahoma, and Tennessee lead; closures cluster heavily in the rural South.
```

**Why this works.** `value_counts().head(10).index` is both the shortlist and the display order, and a `y=` count plot draws it as a clean ranking — the horizontal-bar pattern from Section 5, now on real stakes.

</details>

### Task 2 — Big hospitals or small?

Show the distribution of `beds` for the hospitals that closed: a histogram on top, a boxplot beneath, sharing the x-axis. In a comment, give the median bed count.

In [ ]:
# Your work here


<details>
<summary><b>Solution</b></summary>

```python
fig, axes = plt.subplots(2, 1, figsize=(8, 6), sharex=True)

sns.histplot(closures["beds"], binwidth=5, color="steelblue", ax=axes[0])
axes[0].set_title("Bed counts of closed hospitals")
axes[0].set_ylabel("Hospitals")

sns.boxplot(x=closures["beds"], color="steelblue", ax=axes[1])
axes[1].set_xlabel("Beds")

fig.tight_layout()
plt.show()
# Small: median is 25 beds, and the whole distribution sits under about 55.
```

**Why this works.** This is the histogram-over-boxplot pattern from Notebook 2, pointed at the capstone data. The answer to the driving question is right there in the shape: the hospitals that close are small community hospitals, not large regional ones.

</details>

### Task 3 — Do some payment models lose bigger hospitals?

Plot the **mean `beds` by `payment_type`**, ordered by height, with an honest zero baseline. (The `payment_types.csv` file in the repo documents what each code means.)

In [ ]:
# Your work here


<details>
<summary><b>Solution</b></summary>

```python
order = closures.groupby("payment_type")["beds"].mean().sort_values().index

fig, ax = plt.subplots(figsize=(8, 4))
sns.barplot(data=closures, x="payment_type", y="beds", order=order, color="steelblue", ax=ax)
ax.set_title("Mean beds by Medicare payment type")
ax.set_xlabel("Payment type")
ax.set_ylabel("Mean beds")
fig.tight_layout()
plt.show()
# RRC hospitals that closed are the largest on average (~35 beds); REH the smallest (~19).
```

**Why this works.** It is the ordered aggregate bar from Section 3, on `payment_type`. Because `payment_type` is unordered, sorting by the mean is the right call, and the zero baseline keeps the modest differences honestly modest.

</details>

### One more look — does closure *type* relate to size?

Before the wrap, one comparison worth drawing honestly: do hospitals that close *completely* differ in size from those that *convert* to another kind of facility?

In [ ]:
fig, ax = plt.subplots(figsize=(6, 4))
sns.barplot(data=closures, x="closure_type", y="beds", errorbar=None, color="steelblue", ax=ax)
ax.set_title("Mean beds by closure type (honest baseline)")
ax.set_xlabel("Closure type")
ax.set_ylabel("Mean beds")
fig.tight_layout()
plt.show()

### The insight — and one surprise

**Insight.** Rural closures are not spread evenly. A handful of southern states — Texas, Oklahoma, Tennessee, Alabama — carry a large share, and the hospitals that close are overwhelmingly small, with a median of 25 beds. Geography and size, not payment model, tell most of the story.

**One surprise.** Whether a hospital *closed completely* or *converted* to another kind of facility has almost nothing to do with its size: the mean bed counts are 29.8 and 28.7 — about one bed apart. On the zero baseline above, the two bars look identical, which is the honest reading. It would take exactly the truncated axis you saw in Section 6 to inflate that one-bed gap into a finding — a standing reminder to distrust any version of this chart whose y-axis does not start at zero.

**Next:** Session 2 picks these closures back up — alongside new relationships and time series — and carries them all the way to a presentation-ready figure.